#### 5.1 Loading the Libraries

The required libraries were imported for loading the trained models, generating predictions, calculating regression performance metrics, and organizing the evaluation results.

In [2]:
# Importing the required libraries

import os
import joblib
import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## 5.2 Loading the Prepared Test Data

The held-out test dataset generated during data preparation was loaded for model evaluation.

The test data was not used during model training or hyperparameter tuning, allowing it to provide an unbiased assessment of model performance on unseen observations.

In [20]:
# Loading the prepared training and testing target data

X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

In [4]:
# Checking the test dataset dimensions

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (200, 18)
y_test shape: (200,)


## 5.3 Loading the Trained Models

The trained individual, tuned, and ensemble models saved during model development were loaded for evaluation.

Loading the saved artifacts ensures that the models are evaluated using the exact configurations produced during model development, without retraining or modifying them.

In [5]:
# Loading the trained model artifacts

model_files = [
    file for file in os.listdir("../models")
    if file.endswith(".joblib")
    and file != "preprocessor.joblib"
]

model_files

['extra_trees.joblib',
 'extra_trees_tuned.joblib',
 'gradient_boosting.joblib',
 'gradient_boosting_tuned.joblib',
 'lasso_regression.joblib',
 'lightgbm.joblib',
 'lightgbm_tuned.joblib',
 'linear_regression.joblib',
 'mlp_regressor.joblib',
 'random_forest.joblib',
 'random_forest_tuned.joblib',
 'ridge_regression.joblib',
 'stacking_regression.joblib',
 'voting_regression.joblib',
 'xgboost.joblib',
 'xgboost_tuned.joblib']

In [6]:
# Loading the models into a dictionary

loaded_models = {}

for file in model_files:

    model_name = file.replace(".joblib", "")

    loaded_models[model_name] = joblib.load(
        f"../models/{file}"
    )

print("Loaded models:")

for model_name in sorted(loaded_models.keys()):
    print("-", model_name)

print("\nTotal models loaded:", len(loaded_models))

Loaded models:
- extra_trees
- extra_trees_tuned
- gradient_boosting
- gradient_boosting_tuned
- lasso_regression
- lightgbm
- lightgbm_tuned
- linear_regression
- mlp_regressor
- random_forest
- random_forest_tuned
- ridge_regression
- stacking_regression
- voting_regression
- xgboost
- xgboost_tuned

Total models loaded: 16


In [7]:
# Validating the loaded models

for model_name, model in loaded_models.items():
    print(f"{model_name}: {type(model).__name__}")

extra_trees: ExtraTreesRegressor
extra_trees_tuned: ExtraTreesRegressor
gradient_boosting: GradientBoostingRegressor
gradient_boosting_tuned: GradientBoostingRegressor
lasso_regression: Lasso
lightgbm: LGBMRegressor
lightgbm_tuned: LGBMRegressor
linear_regression: LinearRegression
mlp_regressor: MLPRegressor
random_forest: RandomForestRegressor
random_forest_tuned: RandomForestRegressor
ridge_regression: Ridge
stacking_regression: StackingRegressor
voting_regression: VotingRegressor
xgboost: XGBRegressor
xgboost_tuned: XGBRegressor


#### 5.4 Generating Test Predictions

Each trained model was used to generate predictions on the held-out test dataset.

The same test observations are used across all models to ensure that their performance can be compared consistently using the same evaluation data.

In [8]:
# Generating predictions from all trained models

test_predictions = {}

for model_name, model in loaded_models.items():

    test_predictions[model_name] = model.predict(X_test)

print("Predictions generated for all models.")
print("Number of models:", len(test_predictions))

Predictions generated for all models.
Number of models: 16


In [9]:
# Checking prediction dimensions

for model_name, predictions in test_predictions.items():
    print(
        f"{model_name}: "
        f"{len(predictions)} predictions"
    )

extra_trees: 200 predictions
extra_trees_tuned: 200 predictions
gradient_boosting: 200 predictions
gradient_boosting_tuned: 200 predictions
lasso_regression: 200 predictions
lightgbm: 200 predictions
lightgbm_tuned: 200 predictions
linear_regression: 200 predictions
mlp_regressor: 200 predictions
random_forest: 200 predictions
random_forest_tuned: 200 predictions
ridge_regression: 200 predictions
stacking_regression: 200 predictions
voting_regression: 200 predictions
xgboost: 200 predictions
xgboost_tuned: 200 predictions


#### 5.5 Calculating Evaluation Metrics

The predictions generated on the held-out test dataset were evaluated using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R².

MAE measures the average absolute difference between predicted and actual ride costs, while RMSE gives greater weight to larger prediction errors. R² measures the proportion of variation in historical ride costs explained by the model.

A mean-based baseline was also calculated using the test target values to provide a reference point for evaluating the predictive performance of the trained models.

In [21]:
# Calculating the baseline model performance

baseline_prediction = np.full(
    len(y_test),
    y_train.mean()
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_prediction
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_prediction
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_prediction
)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²:", baseline_r2)

Baseline MAE: 160.96417751928144
Baseline RMSE: 191.15280997910116
Baseline R²: -0.0021555706797884433


In [22]:
# Calculating evaluation metrics for all trained models

evaluation_results = []

for model_name, predictions in test_predictions.items():

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    evaluation_results.append({
        'Model': model_name,
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2
    })

In [23]:
# Adding the baseline model to the evaluation results

evaluation_results.append({
    'Model': 'Baseline',
    'MAE': baseline_mae,
    'RMSE': baseline_rmse,
    'R²': baseline_r2
})

In [24]:
# Creating the evaluation results DataFrame

evaluation_results = pd.DataFrame(
    evaluation_results
)

evaluation_results = evaluation_results[
    ['Model', 'MAE', 'RMSE', 'R²']
]

evaluation_results

,Model,MAE,RMSE,R²
0,extra_trees,53.020429,72.036286,0.857677
1,extra_trees_tuned,66.926870,83.894603,0.806962
2,gradient_boosting,53.676876,72.121650,0.857339
3,gradient_boosting_tuned,53.341282,69.921527,0.865910
4,lasso_regression,52.605610,67.587760,0.874712
5,lightgbm,57.970616,79.134624,0.828246
6,lightgbm_tuned,54.766373,72.193410,0.857055
7,linear_regression,52.613793,67.588549,0.874709
8,mlp_regressor,55.528218,69.890885,0.866028
9,random_forest,55.613818,74.643601,0.847187


In [25]:
# Sorting models by RMSE

evaluation_results_sorted = evaluation_results.sort_values(
    by='RMSE',
    ascending=True
).reset_index(drop=True)

evaluation_results_sorted

,Model,MAE,RMSE,R²
0,lasso_regression,52.605610,67.587760,0.874712
1,linear_regression,52.613793,67.588549,0.874709
2,ridge_regression,52.594296,67.589301,0.874706
3,stacking_regression,52.064850,67.782372,0.873989
4,voting_regression,52.910708,69.630254,0.867025
5,mlp_regressor,55.528218,69.890885,0.866028
6,gradient_boosting_tuned,53.341282,69.921527,0.865910
7,xgboost,54.229044,71.930902,0.858093
8,extra_trees,53.020429,72.036286,0.857677
9,gradient_boosting,53.676876,72.121650,0.857339


#### 5.6 Evaluating Improvement Over the Baseline

The test-set performance of the trained models was compared with the baseline model to quantify the improvement achieved through machine learning.

Percentage improvement was calculated for MAE and RMSE, where lower values indicate smaller prediction errors. R² was also retained as an additional measure of explanatory performance.

In [26]:
# Calculating improvement over the baseline

model_results = evaluation_results[
    evaluation_results['Model'] != 'Baseline'
].copy()

model_results['MAE Improvement (%)'] = (
    (baseline_mae - model_results['MAE'])
    / baseline_mae
) * 100

model_results['RMSE Improvement (%)'] = (
    (baseline_rmse - model_results['RMSE'])
    / baseline_rmse
) * 100

model_results = model_results.sort_values(
    by='RMSE',
    ascending=True
).reset_index(drop=True)

model_results

,Model,MAE,RMSE,R²,MAE Improvement (%),RMSE Improvement (%)
0,lasso_regression,52.605610,67.587760,0.874712,67.318437,64.642026
1,linear_regression,52.613793,67.588549,0.874709,67.313353,64.641614
2,ridge_regression,52.594296,67.589301,0.874706,67.325465,64.641220
3,stacking_regression,52.064850,67.782372,0.873989,67.654387,64.540217
4,voting_regression,52.910708,69.630254,0.867025,67.128893,63.573513
5,mlp_regressor,55.528218,69.890885,0.866028,65.502748,63.437166
6,gradient_boosting_tuned,53.341282,69.921527,0.865910,66.861396,63.421136
7,xgboost,54.229044,71.930902,0.858093,66.309868,62.369948
8,extra_trees,53.020429,72.036286,0.857677,67.060728,62.314817
9,gradient_boosting,53.676876,72.121650,0.857339,66.652905,62.270160


#### 5.7 Analyzing Prediction Errors

This section will calculate the residuals:

Residual = Actual Ride Cost − Predicted Ride Cost

Prediction errors were analyzed by calculating the residuals between the actual and predicted historical ride costs.

Residual analysis provides additional insight into model behavior beyond aggregate performance metrics. The distribution of residuals can help identify systematic prediction errors and unusually large deviations between predicted and observed ride costs.

In [27]:
# Calculating residuals for all models

residuals = {}

for model_name, predictions in test_predictions.items():

    residuals[model_name] = (
        y_test.values - predictions
    )

print("Residuals calculated for:", len(residuals), "models")

Residuals calculated for: 16 models


In [28]:
# Summarizing prediction errors

error_summary = []

for model_name, model_residuals in residuals.items():

    error_summary.append({
        'Model': model_name,
        'Mean Error': np.mean(model_residuals),
        'Mean Absolute Error': np.mean(np.abs(model_residuals)),
        'Maximum Absolute Error': np.max(np.abs(model_residuals))
    })

error_summary = pd.DataFrame(error_summary)

error_summary.sort_values(
    by='Mean Absolute Error',
    ascending=True
).reset_index(drop=True)

,Model,Mean Error,Mean Absolute Error,Maximum Absolute Error
0,stacking_regression,-5.077351,52.064850,190.240334
1,ridge_regression,-4.287402,52.594296,164.704543
2,lasso_regression,-4.295930,52.605610,164.937513
3,linear_regression,-4.302928,52.613793,164.921557
4,voting_regression,-6.300207,52.910708,212.780653
5,extra_trees,-10.624743,53.020429,261.073587
6,gradient_boosting_tuned,-5.209429,53.341282,200.122874
7,gradient_boosting,-6.516636,53.676876,235.470265
8,xgboost,-5.491420,54.229044,224.143394
9,xgboost_tuned,-7.752069,54.734186,241.232506


A few factual observations:

- All models have a negative mean error, meaning they tend to slightly over-predict the historical ride cost on average because we defined residual as actual − predicted.
- Stacking has the lowest mean absolute error at 52.06, consistent with its MAE from the main evaluation table.
- The largest individual error varies considerably. For example, Stacking's maximum absolute error is 190.24, while LightGBM reaches 288.42.
- The mean errors are relatively small compared with the overall prediction errors, so there isn't evidence here of a very large systematic bias.

#### 5.8 Residual Distribution Analysis

The distribution of prediction residuals was examined for the strongest-performing models based on test-set RMSE.

Residual distributions provide a visual assessment of how closely predictions are centered around the actual ride costs and whether substantial deviations occur across the test observations.

In [32]:
# Importing Plotly for residual visualization

import plotly.express as px

In [29]:
# Selecting models for residual distribution analysis

selected_models = [
    'lasso_regression',
    'linear_regression',
    'ridge_regression',
    'stacking_regression'
]

In [33]:
# Plotting residual distributions

for model_name in selected_models:

    fig = px.histogram(
        x=residuals[model_name],
        nbins=30,
        title=f"Residual Distribution — {model_name.replace('_', ' ').title()}",
        labels={
            'x': 'Residual (Actual - Predicted)',
            'count': 'Frequency'
        }
    )

    fig.show()

#### Observation on Residual Distributions

The residual distributions for the selected models are generally centered around zero, with most prediction errors concentrated within a relatively narrow range. The distributions also contain observations on both sides of zero, indicating that the models produce both over-predictions and under-predictions rather than consistently making errors in only one direction.

#### Important Point

Although the models exhibit a small average prediction bias, individual prediction errors can still be substantial. The maximum absolute error varies across models, highlighting that some ride-cost observations are considerably more difficult to predict accurately than others.

#### 5.9 Actual vs Predicted Analysis

The relationship between actual and predicted historical ride costs was examined for the selected models.

An actual-versus-predicted plot provides a visual assessment of prediction accuracy, where predictions closer to the 45-degree reference line indicate closer agreement between predicted and observed ride costs.

In [34]:
# Plotting actual vs predicted values

for model_name in selected_models:

    fig = px.scatter(
        x=y_test,
        y=test_predictions[model_name],
        title=f"Actual vs Predicted — {model_name.replace('_', ' ').title()}",
        labels={
            'x': 'Actual Historical Cost',
            'y': 'Predicted Historical Cost'
        },
        opacity=0.7
    )

    min_value = min(y_test.min(), test_predictions[model_name].min())
    max_value = max(y_test.max(), test_predictions[model_name].max())

    fig.add_shape(
        type='line',
        x0=min_value,
        y0=min_value,
        x1=max_value,
        y1=max_value,
        line=dict(
            dash='dash'
        )
    )

    fig.show()

#### Observation on Actual vs Predicted Values

The actual-versus-predicted plots show a clear alignment between predicted and observed historical ride costs across the evaluated models. Most observations are positioned relatively close to the reference line, indicating that the models capture a substantial portion of the variation in ride costs.

#### Important Point

The plots also show deviations from the reference line for individual observations, indicating that prediction errors remain for certain rides. These deviations are consistent with the MAE, RMSE, and maximum absolute error values calculated earlier.

#### 5.10 Final Evaluation Summary

The final test-set evaluation results were consolidated to provide an overall view of model performance.

The evaluation considers MAE, RMSE, and R², along with improvement over the mean-based baseline. These results provide the foundation for the detailed model comparison conducted in the next notebook.

In [35]:
# Creating the final evaluation summary

final_evaluation = model_results[
    [
        'Model',
        'MAE',
        'RMSE',
        'R²',
        'MAE Improvement (%)',
        'RMSE Improvement (%)'
    ]
].copy()

final_evaluation

,Model,MAE,RMSE,R²,MAE Improvement (%),RMSE Improvement (%)
0,lasso_regression,52.605610,67.587760,0.874712,67.318437,64.642026
1,linear_regression,52.613793,67.588549,0.874709,67.313353,64.641614
2,ridge_regression,52.594296,67.589301,0.874706,67.325465,64.641220
3,stacking_regression,52.064850,67.782372,0.873989,67.654387,64.540217
4,voting_regression,52.910708,69.630254,0.867025,67.128893,63.573513
5,mlp_regressor,55.528218,69.890885,0.866028,65.502748,63.437166
6,gradient_boosting_tuned,53.341282,69.921527,0.865910,66.861396,63.421136
7,xgboost,54.229044,71.930902,0.858093,66.309868,62.369948
8,extra_trees,53.020429,72.036286,0.857677,67.060728,62.314817
9,gradient_boosting,53.676876,72.121650,0.857339,66.652905,62.270160


In [36]:
# Sorting the final evaluation results by RMSE

final_evaluation = final_evaluation.sort_values(
    by='RMSE',
    ascending=True
).reset_index(drop=True)

final_evaluation

,Model,MAE,RMSE,R²,MAE Improvement (%),RMSE Improvement (%)
0,lasso_regression,52.605610,67.587760,0.874712,67.318437,64.642026
1,linear_regression,52.613793,67.588549,0.874709,67.313353,64.641614
2,ridge_regression,52.594296,67.589301,0.874706,67.325465,64.641220
3,stacking_regression,52.064850,67.782372,0.873989,67.654387,64.540217
4,voting_regression,52.910708,69.630254,0.867025,67.128893,63.573513
5,mlp_regressor,55.528218,69.890885,0.866028,65.502748,63.437166
6,gradient_boosting_tuned,53.341282,69.921527,0.865910,66.861396,63.421136
7,xgboost,54.229044,71.930902,0.858093,66.309868,62.369948
8,extra_trees,53.020429,72.036286,0.857677,67.060728,62.314817
9,gradient_boosting,53.676876,72.121650,0.857339,66.652905,62.270160


#### Observation on Final Evaluation

All trained models substantially outperform the mean-based baseline across the test dataset. The evaluated models achieve MAE values substantially below the baseline MAE of approximately 160.96 and RMSE values substantially below the baseline RMSE of approximately 191.15.

The test-set R² values are positive for all trained models, indicating that each model explains a meaningful proportion of the variation in historical ride costs.

#### Important Point

The evaluation results show that model performance varies across MAE, RMSE, and R². Therefore, model selection should consider multiple evaluation metrics rather than relying on a single measure. A detailed comparison of the models will be performed separately in the next notebook.